# Loss Functions and Utility Helpers

This notebook demonstrates the CosmoRecon utility functions and custom loss
classes using small synthetic volumes.

**Utilities**
- `compute_depth` — automatic U-Net depth from grid size
- `shifted_relu` — activation for delta-field mode
- `compute_gradient` — finite-difference 3D spatial gradients
- `dilate_mask` — binary mask dilation via 3D max-pooling
- `prepare_mask_tensor` — reshape helper for mask arrays

**Losses**
- `MaskedMSE` — MSE on missing voxels only
- `MaskedGradientLoss` — gradient matching near hole boundaries
- `MaskedMSEWithGradient` — weighted combination of the above

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from CosmoRecon import (
    compute_depth,
    compute_gradient,
    dilate_mask,
    prepare_mask_tensor,
    shifted_relu,
    MaskedMSE,
    MaskedGradientLoss,
    MaskedMSEWithGradient,
)

---
## Utility functions

### `compute_depth`

The U-Net depth is computed as `floor(log2(min(spatial_dims) / min_size))`.
Larger fields → deeper networks.

In [ ]:
for N in [16, 32, 64, 128, 256]:
    d = compute_depth((N, N, N), min_size=4)
    print(f"  field_size={N:>3d}  ->  depth={d}")

### `shifted_relu`

For delta-field reconstruction, the physical constraint is
`delta >= -1`.  After normalisation by `norm_val`, this becomes
`output >= -1/norm_val`.  `shifted_relu` enforces this.

In [ ]:
x = tf.linspace(-0.2, 0.5, 200)

fig, ax = plt.subplots(figsize=(7, 3.5))
for nv in [5, 20, 40]:
    act = shifted_relu(nv)
    ax.plot(x.numpy(), act(x).numpy(), label=f"norm_val={nv} (min={-1/nv:.3f})")

ax.axhline(0, color="grey", lw=0.5)
ax.set_xlabel("input")
ax.set_ylabel("output")
ax.set_title("shifted_relu for different norm_val")
ax.legend()
plt.tight_layout()
plt.show()

### `compute_gradient`

Finite-difference gradients along each spatial axis of a 5D tensor.  
Each output is one element shorter along the differentiated axis.

In [ ]:
rng = np.random.default_rng(42)
vol = rng.random((1, 16, 16, 16, 1), dtype=np.float32)
vol_tf = tf.constant(vol)

gx, gy, gz = compute_gradient(vol_tf)
print(f"Input shape:  {vol_tf.shape}")
print(f"gx shape:     {gx.shape}  (one fewer along axis 1)")
print(f"gy shape:     {gy.shape}  (one fewer along axis 2)")
print(f"gz shape:     {gz.shape}  (one fewer along axis 3)")

In [ ]:
# Visualise gradients at the central slice
mid = 8
fig, axes = plt.subplots(1, 4, figsize=(14, 3))

axes[0].imshow(vol[0, :, :, mid, 0], origin="lower", cmap="viridis")
axes[0].set_title("Original")

axes[1].imshow(gx[0, :, :, mid, 0].numpy(), origin="lower", cmap="RdBu_r")
axes[1].set_title("Gradient x")

axes[2].imshow(gy[0, :, :, mid, 0].numpy(), origin="lower", cmap="RdBu_r")
axes[2].set_title("Gradient y")

axes[3].imshow(gz[0, :, :15, mid, 0].numpy(), origin="lower", cmap="RdBu_r")
axes[3].set_title("Gradient z")

for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()

### `dilate_mask`

Expands the masked (missing) region using 3D max-pooling.  Each
iteration grows the hole boundary by one voxel.

In [ ]:
N = 16
mask_np = np.ones((N, N, N), dtype=np.float32)
mask_np[6:10, 6:10, 6:10] = 0.0  # small hole

mask_4d = tf.constant(mask_np[np.newaxis, ...])  # (1, N, N, N)

mid = N // 2
fig, axes = plt.subplots(1, 4, figsize=(14, 3))

for i, n_iter in enumerate([0, 1, 2, 3]):
    if n_iter == 0:
        sl = mask_4d[0, :, :, mid].numpy()
    else:
        dilated = dilate_mask(mask_4d, iterations=n_iter)  # (1, N, N, N)
        sl = dilated[0, :, :, mid].numpy()

    axes[i].imshow(sl, origin="lower", cmap="gray_r", vmin=0, vmax=1)
    hole_count = int((sl == 0).sum())
    axes[i].set_title(f"iterations={n_iter}\n(hole voxels: {hole_count})")
    axes[i].set_xticks([]); axes[i].set_yticks([])

fig.suptitle("Mask dilation (white = observed, black = missing)")
plt.tight_layout()
plt.show()

### `prepare_mask_tensor`

Converts masks of various shapes to the canonical `(1, H, W, D, 1)` tensor
for broadcasting.

In [ ]:
for label, arr in [
    ("3D  (H,W,D)",       np.ones((16, 16, 16))),
    ("4D  (1,H,W,D)",     np.ones((1, 16, 16, 16))),
    ("4D  (H,W,D,1)",     np.ones((16, 16, 16, 1))),
    ("5D  (1,H,W,D,1)",   np.ones((1, 16, 16, 16, 1))),
]:
    out = prepare_mask_tensor(arr)
    print(f"  {label:20s}  ->  {out.shape}")

---
## Loss functions

We create a synthetic prediction/target pair and a mask, then evaluate all
three losses to see how they behave.

In [ ]:
N = 16
rng = np.random.default_rng(123)

y_true = rng.random((1, N, N, N, 1), dtype=np.float32)
y_pred = y_true + 0.1 * rng.standard_normal((1, N, N, N, 1)).astype(np.float32)

# Mask with a cubic hole in the centre
mask_np = np.ones((N, N, N), dtype=np.float32)
mask_np[4:12, 4:12, 4:12] = 0.0
mask_tf = prepare_mask_tensor(mask_np)

print(f"y_true shape: {y_true.shape}")
print(f"y_pred shape: {y_pred.shape}")
print(f"mask shape:   {mask_tf.shape}")
print(f"Fraction missing: {1 - mask_np.mean():.2%}")

### `MaskedMSE`

MSE restricted to the missing region (where `mask == 0`).

In [ ]:
loss_mse = MaskedMSE(mask=mask_tf)
val_mse = loss_mse(y_true, y_pred).numpy()

# Compare with plain MSE on the same missing region
inv_mask = 1.0 - mask_np[np.newaxis, ..., np.newaxis]
manual_mse = np.sum((y_true - y_pred)**2 * inv_mask) / np.sum(inv_mask)

print(f"MaskedMSE loss:  {val_mse:.6f}")
print(f"Manual check:    {manual_mse:.6f}")
print(f"Match: {np.isclose(val_mse, manual_mse, atol=1e-5)}")

### `MaskedGradientLoss`

Penalises gradient differences near hole boundaries.  A `dilation_iter`
of 2 means the boundary region extends 2 voxels outward from each hole edge.

In [ ]:
loss_grad = MaskedGradientLoss(mask=mask_tf, dilation_iter=2)
val_grad = loss_grad(y_true, y_pred).numpy()
print(f"MaskedGradientLoss: {val_grad:.6f}")

# More dilation iterations -> larger boundary region -> potentially larger loss
for d_iter in [1, 2, 3]:
    lg = MaskedGradientLoss(mask=mask_tf, dilation_iter=d_iter)
    print(f"  dilation_iter={d_iter}: {lg(y_true, y_pred).numpy():.6f}")

### `MaskedMSEWithGradient`

Weighted combination: `total = MaskedMSE + gradient_weight * MaskedGradientLoss`.

In [ ]:
for gw in [0.01, 0.1, 0.5, 1.0]:
    loss_combined = MaskedMSEWithGradient(mask=mask_tf, gradient_weight=gw)
    val = loss_combined(y_true, y_pred).numpy()
    expected = val_mse + gw * val_grad
    print(f"  gradient_weight={gw:.2f}  ->  loss={val:.6f}  (expected {expected:.6f})")

### `set_mask()` for model reloading

When a model with a masked loss is saved and reloaded, the mask is not
serialised.  You must call `set_mask()` on the loss to restore it before
resuming training.

In [ ]:
# Simulate: create a loss without a mask (as from_config would)
loss_empty = MaskedMSE(mask=None)

try:
    loss_empty(y_true, y_pred)
except RuntimeError as e:
    print(f"Expected error: {e}")

# Re-inject the mask
loss_empty.set_mask(mask_tf)
val = loss_empty(y_true, y_pred).numpy()
print(f"After set_mask(): loss = {val:.6f}  (matches original: {np.isclose(val, val_mse, atol=1e-5)})")